In [1]:

# ============================================================
# RAILPULSE AI — FULL CURRENT ARRIVAL DELAY AUDIT
# ============================================================

import os
import gc
import ijson
import numpy as np
import pandas as pd

JSON_PATH = (
    "../data/interim/RSTGCN/"
    "Indian-Railway-Network-and-Delays/"
    "train_delays_Sep2024.json"
)

OUTPUT_DIR = "../outputs/tables"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ------------------------------------------------------------
# 1. TIME PARSING HELPERS
# ------------------------------------------------------------

def time_to_minutes(value):

    if not value:
        return np.nan

    try:
        parsed = pd.to_datetime(
            value,
            format="%I:%M %p"
        )

        return parsed.hour * 60 + parsed.minute

    except Exception:
        return np.nan


def delay_to_minutes(value):

    if value is None:
        return np.nan

    text = str(value).strip().upper()

    if not text:
        return np.nan

    import re

    hours = re.search(r"(\d+)\s*H", text)
    minutes = re.search(r"(\d+)\s*M", text)

    h = int(hours.group(1)) if hours else 0
    m = int(minutes.group(1)) if minutes else 0

    return h * 60 + m


# ------------------------------------------------------------
# 2. READ ORIGINAL JSON STREAMING
# ------------------------------------------------------------

audit_records = []

print("=" * 75)
print("RAILPULSE AI — FULL ARRIVAL DELAY AUDIT")
print("=" * 75)

with open(JSON_PATH, "rb") as f:

    for train, train_data in ijson.kvitems(f, ""):

        for date, station_data in train_data.items():

            if not isinstance(station_data, dict):
                continue

            for station, values in station_data.items():

                if not isinstance(values, list) or len(values) < 3:
                    continue

                scheduled = time_to_minutes(values[0])
                actual = time_to_minutes(values[1])
                reported = delay_to_minutes(values[2])

                if (
                    np.isnan(scheduled)
                    or np.isnan(actual)
                    or np.isnan(reported)
                ):
                    continue

                clock_difference = (
                    actual - scheduled
                ) % 1440

                discrepancy = (
                    reported - clock_difference
                )

                audit_records.append({
                    "train": str(train),
                    "date": date,
                    "station": station,
                    "scheduled_arrival": values[0],
                    "actual_arrival": values[1],
                    "reported_delay": reported,
                    "clock_delay_mod_1440": clock_difference,
                    "discrepancy": discrepancy
                })

        if len(audit_records) > 500_000:
            print(
                "Parsed records:",
                f"{len(audit_records):,}"
            )

# ------------------------------------------------------------
# 3. BUILD AUDIT DATAFRAME
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_records)

del audit_records
gc.collect()

print("\nTotal valid timestamp records:", len(audit_df))

# ------------------------------------------------------------
# 4. DISCREPANCY SUMMARY
# ------------------------------------------------------------

audit_df["absolute_discrepancy"] = (
    audit_df["discrepancy"].abs()
)

thresholds = [1, 5, 15, 30, 60, 120, 720, 1440]

print("\n" + "=" * 75)
print("ARRIVAL DELAY DISCREPANCY SUMMARY")
print("=" * 75)

print(
    "Exact matches:",
    (audit_df["absolute_discrepancy"] == 0).sum()
)

print(
    "Non-matches:",
    (audit_df["absolute_discrepancy"] > 0).sum()
)

for threshold in thresholds:

    count = (
        audit_df["absolute_discrepancy"] >= threshold
    ).sum()

    percentage = count / len(audit_df) * 100

    print(
        f"Discrepancy >= {threshold:4} min: "
        f"{count:9,} ({percentage:.3f}%)"
    )

# ------------------------------------------------------------
# 5. DISCREPANCY DISTRIBUTION
# ------------------------------------------------------------

print("\nDiscrepancy statistics:")

print(
    audit_df["discrepancy"].describe(
        percentiles=[
            0.25, 0.50, 0.75, 0.90, 0.95, 0.99
        ]
    ).round(2)
)

# ------------------------------------------------------------
# 6. LARGEST DISCREPANCIES
# ------------------------------------------------------------

largest = audit_df.sort_values(
    "absolute_discrepancy",
    ascending=False
).head(50)

print("\nTOP 50 ARRIVAL DELAY DISCREPANCIES")

print(
    largest.to_string(
        index=False
    )
)

# ------------------------------------------------------------
# 7. EXACT DAY-OFFSET CHECK
# ------------------------------------------------------------

day_offset = (
    audit_df["discrepancy"].abs() % 1440
) == 0

print("\nExact multiples of 1440 minutes:")
print(
    int(
        (
            audit_df["absolute_discrepancy"] > 0
        ).mul(day_offset).sum()
    )
)

# ------------------------------------------------------------
# 8. SAVE OUTPUTS
# ------------------------------------------------------------

audit_df.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "full_arrival_delay_quality_audit.csv"
    ),
    index=False
)

largest.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "top_50_arrival_delay_discrepancies.csv"
    ),
    index=False
)

print("\nSaved:")
print("full_arrival_delay_quality_audit.csv")
print("top_50_arrival_delay_discrepancies.csv")

print("\nFULL ARRIVAL DELAY AUDIT COMPLETED.")

gc.collect()


RAILPULSE AI — FULL ARRIVAL DELAY AUDIT
Parsed records: 500,441
Parsed records: 500,517
Parsed records: 500,725
Parsed records: 500,771
Parsed records: 500,906
Parsed records: 500,942
Parsed records: 501,272
Parsed records: 501,392
Parsed records: 501,632
Parsed records: 501,704
Parsed records: 502,488
Parsed records: 502,507
Parsed records: 502,843
Parsed records: 503,263
Parsed records: 503,379
Parsed records: 503,478
Parsed records: 503,694
Parsed records: 503,854
Parsed records: 503,920
Parsed records: 503,956
Parsed records: 503,992
Parsed records: 504,820
Parsed records: 504,950
Parsed records: 505,067
Parsed records: 505,307
Parsed records: 505,328
Parsed records: 506,110
Parsed records: 507,520
Parsed records: 507,970
Parsed records: 509,410
Parsed records: 509,635
Parsed records: 509,665
Parsed records: 509,709
Parsed records: 511,711
Parsed records: 511,903
Parsed records: 512,353
Parsed records: 513,084
Parsed records: 513,588
Parsed records: 513,966
Parsed records: 514,070


0

In [5]:

from pathlib import Path
import pandas as pd

audit_path = Path("../outputs/tables/full_arrival_delay_quality_audit.csv")
discrepancy_path = Path("../outputs/tables/top_50_arrival_delay_discrepancies.csv")

# Load full audit summary
audit = pd.read_csv(audit_path)

# Load top 50 discrepancies
discrepancies = pd.read_csv(discrepancy_path)

print("=" * 70)
print("FULL ARRIVAL DELAY QUALITY AUDIT")
print("=" * 70)

print("Shape:", audit.shape)
print("Columns:", audit.columns.tolist())
display(audit.head(15))

print("\n" + "=" * 70)
print("TOP 50 ARRIVAL DELAY DISCREPANCIES")
print("=" * 70)

print("Shape:", discrepancies.shape)
print("Columns:", discrepancies.columns.tolist())
display(discrepancies.head(15))


FULL ARRIVAL DELAY QUALITY AUDIT
Shape: (1282325, 9)
Columns: ['train', 'date', 'station', 'scheduled_arrival', 'actual_arrival', 'reported_delay', 'clock_delay_mod_1440', 'discrepancy', 'absolute_discrepancy']


,train,date,station,scheduled_arrival,actual_arrival,reported_delay,clock_delay_mod_1440,discrepancy,absolute_discrepancy
0,12303,02-09-2024,HWH,08:00 AM,08:01 AM,0,1,-1,1
1,12303,02-09-2024,BWN,09:05 AM,09:20 AM,15,15,0,0
2,12303,02-09-2024,DGR,09:57 AM,10:00 AM,3,3,0,0
3,12303,02-09-2024,ASN,10:32 AM,10:34 AM,2,2,0,0
4,12303,02-09-2024,CRJ,11:00 AM,11:04 AM,4,4,0,0
5,12303,02-09-2024,JMT,11:15 AM,11:17 AM,2,2,0,0
6,12303,02-09-2024,MDP,11:44 AM,11:46 AM,2,2,0,0
7,12303,02-09-2024,JSME,12:11 PM,12:11 PM,0,0,0,0
8,12303,02-09-2024,JAJ,01:12 PM,01:12 PM,0,0,0,0
9,12303,02-09-2024,JMU,01:33 PM,01:37 PM,4,4,0,0



TOP 50 ARRIVAL DELAY DISCREPANCIES
Shape: (50, 9)
Columns: ['train', 'date', 'station', 'scheduled_arrival', 'actual_arrival', 'reported_delay', 'clock_delay_mod_1440', 'discrepancy', 'absolute_discrepancy']


,train,date,station,scheduled_arrival,actual_arrival,reported_delay,clock_delay_mod_1440,discrepancy,absolute_discrepancy
0,13484,22-09-2024,EKI,10:20 AM,10:22 AM,4322,2,4320,4320
1,13484,22-09-2024,BNDP,10:55 AM,11:11 AM,4336,16,4320,4320
2,13484,22-09-2024,GRMP,11:12 AM,11:25 AM,4333,13,4320,4320
3,12601,20-09-2024,JTJ,11:23 PM,12:14 AM,2931,51,2880,2880
4,12129,18-09-2024,HWH,04:15 AM,10:05 AM,1790,350,1440,1440
5,22645,02-09-2024,SRTL,11:34 AM,01:18 PM,1544,104,1440,1440
6,22645,02-09-2024,ALLP,11:57 AM,01:52 PM,1555,115,1440,1440
7,12129,18-09-2024,JSG,07:50 PM,11:23 PM,1653,213,1440,1440
8,12129,18-09-2024,TATA,11:42 PM,05:05 AM,1763,323,1440,1440
9,12616,02-09-2024,KMT,07:18 PM,11:54 PM,1716,276,1440,1440


In [6]:

import pandas as pd
from pathlib import Path

audit_path = Path("../outputs/tables/full_arrival_delay_quality_audit.csv")

audit = pd.read_csv(audit_path)

print("TOTAL RECORDS:", len(audit))
print("\nDISCREPANCY DISTRIBUTION:")

audit["absolute_discrepancy"] = audit["absolute_discrepancy"].abs()

print(
    audit["absolute_discrepancy"]
    .value_counts()
    .sort_index()
    .tail(15)
)

print("\nTHRESHOLD SUMMARY:")

for threshold in [0, 1, 5, 30, 60, 1440]:
    count = (audit["absolute_discrepancy"] >= threshold).sum()
    print(f"Discrepancy >= {threshold:>4} minutes: {count:,}")

major = audit[audit["absolute_discrepancy"] >= 5]

print("\nMAJOR DISCREPANCIES")
print("Records:", len(major))
print("Affected trains:", major["train"].nunique())
print("Affected stations:", major["station"].nunique())
print("Affected dates:", major["date"].nunique())

print("\nDiscrepancy values:")
print(major["discrepancy"].value_counts().sort_index())


TOTAL RECORDS: 1282325

DISCREPANCY DISTRIBUTION:
absolute_discrepancy
0       1222878
1         59262
1440        181
2880          1
4320          3
Name: count, dtype: int64

THRESHOLD SUMMARY:
Discrepancy >=    0 minutes: 1,282,325
Discrepancy >=    1 minutes: 59,447
Discrepancy >=    5 minutes: 185
Discrepancy >=   30 minutes: 185
Discrepancy >=   60 minutes: 185
Discrepancy >= 1440 minutes: 185

MAJOR DISCREPANCIES
Records: 185
Affected trains: 22
Affected stations: 116
Affected dates: 17

Discrepancy values:
discrepancy
1440    181
2880      1
4320      3
Name: count, dtype: int64


In [10]:

import pandas as pd
from pathlib import Path

# Normalize dates in the audit data
flagged = audit.loc[
    audit["absolute_discrepancy"] >= 5,
    ["train", "date", "station"]
].copy()

flagged["train"] = flagged["train"].astype(str).str.strip()
flagged["date"] = pd.to_datetime(
    flagged["date"], format="%d-%m-%Y"
).dt.strftime("%Y-%m-%d")
flagged["station"] = flagged["station"].astype(str).str.strip()

flagged = flagged.drop_duplicates(
    ["train", "date", "station"]
)

current_flags = flagged.rename(
    columns={"station": "current_station"}
)

next_flags = flagged.rename(
    columns={"station": "next_station"}
)

file_path = Path("../data/processed/railpulse_model_features.csv")

current_count = 0
next_count = 0
both_count = 0

for chunk in pd.read_csv(
    file_path,
    usecols=[
        "train", "date", "current_station", "next_station"
    ],
    dtype=str,
    chunksize=50000
):
    for col in [
        "train", "date", "current_station", "next_station"
    ]:
        chunk[col] = chunk[col].str.strip()

    # Match flagged current stations
    current_hits = chunk.merge(
        current_flags,
        on=["train", "date", "current_station"],
        how="inner"
    )

    # Match flagged next stations
    next_hits = chunk.merge(
        next_flags,
        on=["train", "date", "next_station"],
        how="inner"
    )

    # Match rows where both stations are flagged
    both_hits = current_hits.merge(
        next_flags,
        on=["train", "date", "next_station"],
        how="inner"
    )

    current_count += len(current_hits)
    next_count += len(next_hits)
    both_count += len(both_hits)

print("=" * 65)
print("RAILPULSE AI — CORRECTED MODEL DATA IMPACT AUDIT")
print("=" * 65)
print("Major discrepancy records:", len(flagged))
print("Affected model rows — current station:", current_count)
print("Affected model rows — next station:", next_count)
print("Rows with both stations affected:", both_count)
print("Total model dataset rows: 1,101,443")
print("=" * 65)


RAILPULSE AI — CORRECTED MODEL DATA IMPACT AUDIT
Major discrepancy records: 185
Affected model rows — current station: 157
Affected model rows — next station: 167
Rows with both stations affected: 141
Total model dataset rows: 1,101,443


In [8]:

import pandas as pd

model_path = "../data/processed/railpulse_model_features.csv"

model_sample = pd.read_csv(
    model_path,
    usecols=["train", "date", "current_station", "next_station"],
    nrows=5
)

print("AUDIT DATE EXAMPLES:")
print(audit.loc[audit["absolute_discrepancy"] >= 5, "date"].head().tolist())

print("\nMODEL DATE EXAMPLES:")
print(model_sample["date"].tolist())

print("\nAUDIT TRAIN TYPE:", audit["train"].dtype)
print("MODEL TRAIN TYPE:", model_sample["train"].dtype)

print("\nMODEL STATION EXAMPLES:")
print(model_sample[["current_station", "next_station"]].to_string(index=False))


AUDIT DATE EXAMPLES:
['05-09-2024', '05-09-2024', '05-09-2024', '05-09-2024', '05-09-2024']

MODEL DATE EXAMPLES:
['2024-09-01', '2024-09-01', '2024-09-01', '2024-09-01', '2024-09-01']

AUDIT TRAIN TYPE: int64
MODEL TRAIN TYPE: int64

MODEL STATION EXAMPLES:
current_station next_station
           CSMT           DR
             DR          TNA
            TNA         PNVL
           PNVL          MNI
            MNI         KHED


In [11]:

import pandas as pd
from pathlib import Path

# Load the full timestamp audit
audit = pd.read_csv(
    "../outputs/tables/full_arrival_delay_quality_audit.csv",
    dtype={"train": str}
)

# Identify major discrepancies (5 minutes or more)
flagged = audit.loc[
    audit["absolute_discrepancy"] >= 5,
    ["train", "date", "station"]
].copy()

flagged["train"] = flagged["train"].str.strip()
flagged["date"] = pd.to_datetime(
    flagged["date"], format="%d-%m-%Y"
).dt.strftime("%Y-%m-%d")
flagged["station"] = flagged["station"].str.strip()

flagged = flagged.drop_duplicates(
    ["train", "date", "station"]
)

current_flags = flagged.rename(
    columns={"station": "current_station"}
)

next_flags = flagged.rename(
    columns={"station": "next_station"}
)

# Scan the model dataset in memory-safe chunks
feature_path = Path(
    "../data/processed/railpulse_model_features.csv"
)

affected_chunks = []

for chunk in pd.read_csv(
    feature_path,
    usecols=[
        "train", "date",
        "current_station", "next_station"
    ],
    dtype=str,
    chunksize=50000
):
    for col in [
        "train", "date",
        "current_station", "next_station"
    ]:
        chunk[col] = chunk[col].str.strip()

    current_hits = chunk.merge(
        current_flags,
        on=["train", "date", "current_station"],
        how="inner"
    )

    next_hits = chunk.merge(
        next_flags,
        on=["train", "date", "next_station"],
        how="inner"
    )

    affected = pd.concat(
        [current_hits, next_hits],
        ignore_index=True
    )

    affected_chunks.append(
        affected[
            ["train", "date", "current_station", "next_station"]
        ]
    )

affected_rows = pd.concat(
    affected_chunks,
    ignore_index=True
).drop_duplicates()

# Save the exact affected model rows
output_path = Path(
    "../outputs/tables/affected_model_rows.csv"
)

affected_rows.to_csv(output_path, index=False)

print("SENSITIVITY AUDIT PREPARATION")
print("=" * 50)
print("Unique affected model rows:", len(affected_rows))
print("Saved to:", output_path.resolve())


SENSITIVITY AUDIT PREPARATION
Unique affected model rows: 183
Saved to: C:\Users\Login\Documents\RailPulse_AI\outputs\tables\affected_model_rows.csv
